In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
SFT_PATH       = '/content/drive/MyDrive/ns_proj/sft_train.jsonl'
MODEL_OUT_PATH = '/content/drive/MyDrive/ns_proj/qwen3_lora'
GGUF_OUT_PATH  = '/content/drive/MyDrive/ns_proj/qwen3_finetuned.gguf'

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wj75srga/unsloth_eae1c6195e9e4240938d35ebf4001e84
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wj75srga/unsloth_eae1c6195e9e4240938d35ebf4001e84
  Resolved https://github.com/unslothai/unsloth.git to commit 0ad814a45228999d95857f8e38a484f9ce107c92
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.3 MB/s eta 0:00:00
   ━

In [ ]:
import json

with open(SFT_PATH) as f:
    lines = [l for l in f if l.strip()]
print(f"{len(lines)} rows\n")

row = json.loads(lines[0])
for k, v in row.items():
    print(f"{k:16} :: {type(v).__name__:4} :: {repr(v)[:160]}")

915 rows

run_id           :: str  :: '0ee93d4e21664afca75896231753f07b'
problem_text     :: str  :: 'There are 4 positions numbered 1 through 4. Helena, Luther, Thomas, and Quinn occupy these positions. Quinn is before Helena. Helena is immediately before Luth
active_domains   :: str  :: '["ordering"]'
extracted_json   :: str  :: '{"entities": ["Helena", "Luther", "Thomas", "Quinn"], "constraints": [{"type": "before", "left": "Quinn", "right": "Helena", "evidence_text": "Quinn is before 
model_name       :: str  :: 'algorithmic_gemini-3.1-flash-lite'
timestamp        :: str  :: '2026-06-27T18:26:51.010449+00:00'


In [ ]:
from datasets import Dataset

raw = [json.loads(l) for l in lines]

SYSTEM = "Extract logic puzzles into JSON. Return ONLY a JSON object, no explanation."

def to_messages(row):
    domains   = json.loads(row["active_domains"])              # 2nd decode
    user      = (f"Active domains: {', '.join(domains)}\n\n"
                 f"Extract this logic puzzle:\n\n{row['problem_text']}")
    assistant = row["extracted_json"]                          # already canonical JSON
    return [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]

ds = Dataset.from_list([{
    "messages":       to_messages(r),
    "problem_text":   r["problem_text"],
    "active_domains": r["active_domains"],
    "extracted_json": r["extracted_json"],
} for r in raw])

print(len(ds), "SFT examples")
print(ds[0]["messages"][0]["content"][:400])
print("======================================")
print(ds[0]["messages"][1]["content"][:400])
print("===========================================")
print(ds[0]["messages"][2]["content"][:400])
print("===========================================")
print("ds[0].keys():")
print(ds[0].keys())


915 SFT examples
Extract logic puzzles into JSON. Return ONLY a JSON object, no explanation.
Active domains: ordering

Extract this logic puzzle:

There are 4 positions numbered 1 through 4. Helena, Luther, Thomas, and Quinn occupy these positions. Quinn is before Helena. Helena is immediately before Luther. Helena and Luther are adjacent. Helena and Quinn are adjacent. Thomas and Quinn are adjacent. If Thomas and Quinn are adjacent, then Quinn is before Luther. Which of the following is 
{"entities": ["Helena", "Luther", "Thomas", "Quinn"], "constraints": [{"type": "before", "left": "Quinn", "right": "Helena", "evidence_text": "Quinn is before Helena."}, {"type": "immediately_before", "left": "Helena", "right": "Luther", "evidence_text": "Helena is immediately before Luther."}, {"type": "adjacent", "left": "Helena", "right": "Luther", "evidence_text": "Helena and Luther are adjace
ds[0].keys():
dict_keys(['messages', 'problem_text', 'active_domains', 'extracted_json'])


In [ ]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-0.6B",
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

def render(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False,
            add_generation_prompt=False, enable_thinking=False)
        for m in batch["messages"]
    ]}

ds = ds.map(render, batched=True)
print(ds[0]["text"][:1200])   # <-- check assistant span is bare JSON

==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/915 [00:00<?, ? examples/s]

<|im_start|>system
Extract logic puzzles into JSON. Return ONLY a JSON object, no explanation.<|im_end|>
<|im_start|>user
Active domains: ordering

Extract this logic puzzle:

There are 4 positions numbered 1 through 4. Helena, Luther, Thomas, and Quinn occupy these positions. Quinn is before Helena. Helena is immediately before Luther. Helena and Luther are adjacent. Helena and Quinn are adjacent. Thomas and Quinn are adjacent. If Thomas and Quinn are adjacent, then Quinn is before Luther. Which of the following is correct?
A) [could be true] Quinn is in slot 2
B) [could be true] Quinn is in slot 4<|im_end|>
<|im_start|>assistant
<think>

</think>

{"entities": ["Helena", "Luther", "Thomas", "Quinn"], "constraints": [{"type": "before", "left": "Quinn", "right": "Helena", "evidence_text": "Quinn is before Helena."}, {"type": "immediately_before", "left": "Helena", "right": "Luther", "evidence_text": "Helena is immediately before Luther."}, {"type": "adjacent", "left": "Helena", "right"

In [ ]:
# ── SPLIT ──────────────────────────────────────────────
ds = ds.train_test_split(test_size=0.15, seed=42)
ds_train = ds["train"]
ds_eval  = ds["test"]
# ── FILTER ─────────────────────────────────────────────
def within_budget(batch):
    lengths = [len(ids) for ids in tokenizer(batch["text"])["input_ids"]]
    return [l <= MAX_SEQ for l in lengths]

ds_train = ds_train.filter(within_budget, batched=True)
ds_eval  = ds_eval.filter(within_budget, batched=True)
print(f"after filter  train={len(ds_train)}  eval={len(ds_eval)}")

Filter:   0%|          | 0/777 [00:00<?, ? examples/s]

Filter:   0%|          | 0/138 [00:00<?, ? examples/s]

after filter  train=777  eval=138


In [ ]:
from __future__ import annotations
from pydantic import BaseModel, create_model, Field, ConfigDict
from typing import Annotated, Union, Optional, Literal


class BinaryOrdering(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["before", "immediately_before", "adjacent"]   # was "not_adjacent"
    left: str
    right: str
    evidence_text: Optional[str] = None

class SlotFixed(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["slot_fixed"]
    entity: str
    slot: int
    evidence_text: Optional[str] = None

class KKConstraint(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["is_truth_teller", "is_deceiver"]
    entity: str
    evidence_text: Optional[str] = None

class GroupRelation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["same_group", "different_group"]
    entities: list[str]
    evidence_text: Optional[str] = None

class ExactlyN(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["exactly_n"]
    entities: list[str]
    n: int
    group: Optional[int] = None
    evidence_text: Optional[str] = None

class IsIn(BaseModel):                                           # NEW class
    model_config = ConfigDict(extra="forbid")
    type: Literal["is_in"]
    entity: str
    group: int
    evidence_text: Optional[str] = None

DOMAIN_CONSTRAINT_CLASSES = {
    "ordering":           [BinaryOrdering, SlotFixed],
    "knights_and_knaves": [KKConstraint],
    "grouping":           [GroupRelation, ExactlyN, IsIn],       # added IsIn
}

DOMAIN_LP_FIELDS = {
    "ordering":           {"num_slots":   (int, ...)},
    "grouping":           {"num_groups":  (int, ...)},
    "knights_and_knaves": {},
}

def build_hybrid_schema(active_domains: list[str]) -> type:
    leaf_classes = [cls for d in active_domains for cls in DOMAIN_CONSTRAINT_CLASSES[d]]

    IfThen = create_model("IfThen", __config__=ConfigDict(extra="forbid"),
        type=(Literal["if_then"], ...),
        antecedent=("HybridConstraint", ...),
        consequent=("HybridConstraint", ...),
        evidence_text=(Optional[str], None))
    NotC = create_model("NotC", __config__=ConfigDict(extra="forbid"),
        type=(Literal["not"], ...),
        claim=("HybridConstraint", ...),
        evidence_text=(Optional[str], None))
    AndOr = create_model("AndOr", __config__=ConfigDict(extra="forbid"),
        type=(Literal["and", "or"], ...),
        claims=(list["HybridConstraint"], ...),
        evidence_text=(Optional[str], None))

    members = tuple(leaf_classes) + (IfThen, NotC, AndOr)
    HybridConstraint = Annotated[Union[members], Field(discriminator="type")]

    namespace = {"HybridConstraint": HybridConstraint}
    for w in (IfThen, NotC, AndOr):
        w.model_rebuild(_types_namespace=namespace)

    AnswerChoice = create_model("AnswerChoice",
        label=(str, ...),
        type=(Literal["must_be_true", "could_be_true", "must_be_false", "could_be_false"], ...),
        constraints=(list[HybridConstraint], ...))
    Question = create_model("Question",
        question_constraints=(list[HybridConstraint], ...),
        answer_choices=(list[AnswerChoice], ...))

    base_fields = {
        "entities":    (list[str], ...),
        "constraints": (list[HybridConstraint], ...),
        "questions":   (list[Question], ...),
    }
    for d in active_domains:
        base_fields.update(DOMAIN_LP_FIELDS[d])

    return create_model("LogicProblem", **base_fields)


In [ ]:
!pip install z3-solver
from z3 import (Solver, Int, Bool, Distinct, Sum, If, Implies,
                Not, And, Or, BoolVal, sat, unsat, Abs)

ORDERING_TYPES = {"before", "immediately_before", "adjacent", "slot_fixed"}   # was not_adjacent
GROUPING_TYPES = {"same_group", "different_group", "exactly_n", "is_in"}       # added is_in
KK_TYPES       = {"is_truth_teller", "is_deceiver"}

def z3_solve(extracted) -> dict:
    solver = Solver()

    # -------------------------------------------------------------------------
    # 1. Build vars from types actually present in extracted constraints
    #    — not from active_domains, guarding against classifier hallucination
    # -------------------------------------------------------------------------
    all_types = _used_types(extracted)
    vars = {}

    if all_types & ORDERING_TYPES:
        vars.update({f"slot_{e}": Int(f"slot_{e}") for e in extracted.entities})
        solver.add(Distinct([vars[f"slot_{e}"] for e in extracted.entities]))
        for e in extracted.entities:
            solver.add(vars[f"slot_{e}"] >= 1, vars[f"slot_{e}"] <= extracted.num_slots)

    if all_types & GROUPING_TYPES:
        vars.update({f"group_{e}": Int(f"group_{e}") for e in extracted.entities})
        for e in extracted.entities:
            solver.add(vars[f"group_{e}"] >= 1, vars[f"group_{e}"] <= extracted.num_groups)
        # REMOVED: THE group_sizes ENFORCEMENT LOOP THAT ASSERTED
        #          Sum(entities in group g) == extracted.group_sizes[g_idx]
        #

    if all_types & KK_TYPES:
        vars.update({f"kk_{e}": Bool(f"kk_{e}") for e in extracted.entities})

    # -------------------------------------------------------------------------
    # 2. Add problem-level constraints with tracking for unsat core
    # -------------------------------------------------------------------------
    trackers = {}
    for i, c in enumerate(extracted.constraints):
        tracker = Bool(f"track_c_{i}")
        trackers[tracker] = format_constraint(c)
        solver.assert_and_track(encode(c, vars), tracker)

    # -------------------------------------------------------------------------
    # 3. Check base problem level satisfiability
    # -------------------------------------------------------------------------
    status = solver.check()
    if status == unsat:
        core = solver.unsat_core()
        core_descriptions = [trackers[b] for b in core if b in trackers]
        return {"status": "unsat", "unsat_core": core_descriptions}
    if status != sat:
        return {"status": "unknown", "answer": None}

    # -------------------------------------------------------------------------
    # 4. Verify each question independently via push/pop
    # CHANGED: was a single question, now loops over extracted.questions
    # push/pop means base constraints are built once and reused across all questions
    # -------------------------------------------------------------------------
    question_results = []

    for question in extracted.questions:
        solver.push()
        for qc in question.question_constraints:    # narrows solution space for this question only
            solver.add(encode(qc, vars))            # popped with outer solver.pop() after choices

        choice_results = {}
        for choice in question.answer_choices:
            solver.push()

            choice_expr = (
                And(*[encode(c, vars) for c in choice.constraints])
                if choice.constraints
                else BoolVal(True)
            )

            if   choice.type == "must_be_true":   solver.add(Not(choice_expr));  verified = solver.check() == unsat
            elif choice.type == "could_be_true":  solver.add(choice_expr);       verified = solver.check() == sat
            elif choice.type == "must_be_false":  solver.add(choice_expr);       verified = solver.check() == unsat
            elif choice.type == "could_be_false": solver.add(Not(choice_expr));  verified = solver.check() == sat
            else: verified = False

            choice_results[choice.label] = verified
            solver.pop()

        correct = [label for label, verified in choice_results.items() if verified]
        question_results.append(choice_results) #there is 1 choice_results per question

        solver.pop()

    return {
        "status":           "sat",
        "question_results": question_results
    }


def format_constraint(c) -> str:
    t = c.type
    if   t == "before":             return f"before({c.left}, {c.right})"
    elif t == "immediately_before": return f"immediately_before({c.left}, {c.right})"
    elif t == "adjacent":           return f"adjacent({c.left}, {c.right})"            # was not_adjacent
    elif t == "slot_fixed":         return f"slot_fixed({c.entity}, slot={c.slot})"
    elif t == "is_truth_teller":    return f"is_truth_teller({c.entity})"
    elif t == "is_deceiver":        return f"is_deceiver({c.entity})"
    elif t == "same_group":         return f"same_group({', '.join(c.entities)})"
    elif t == "different_group":    return f"different_group({', '.join(c.entities)})"
    elif t == "exactly_n":          return f"exactly_n({', '.join(c.entities)}, n={c.n}, group={c.group})"
    elif t == "is_in":              return f"is_in({c.entity}, group={c.group})"        # NEW
    elif t == "if_then":            return f"if_then({format_constraint(c.antecedent)}, {format_constraint(c.consequent)})"
    elif t == "not":                return f"not({format_constraint(c.claim)})"
    elif t == "and":                return f"and({', '.join(format_constraint(cl) for cl in c.claims)})"
    elif t == "or":                 return f"or({', '.join(format_constraint(cl) for cl in c.claims)})"
    else:                           return f"unknown({t})"


def encode(c, vars: dict):
    t = c.type

    # Ordering
    if   t == "before":             return vars[f"slot_{c.left}"] < vars[f"slot_{c.right}"]
    elif t == "immediately_before": return vars[f"slot_{c.left}"] + 1 == vars[f"slot_{c.right}"]
    elif t == "adjacent":           return Abs(vars[f"slot_{c.left}"] - vars[f"slot_{c.right}"]) == 1   # was not_adjacent: Abs(...) > 1
    elif t == "slot_fixed":         return vars[f"slot_{c.entity}"] == c.slot

    # Knights and Knaves
    elif t == "is_truth_teller":    return vars[f"kk_{c.entity}"]
    elif t == "is_deceiver":        return Not(vars[f"kk_{c.entity}"])

    # Grouping
    elif t == "same_group":
        first = vars[f"group_{c.entities[0]}"]
        return And(*[vars[f"group_{e}"] == first for e in c.entities[1:]])
    elif t == "different_group":
        return Distinct([vars[f"group_{e}"] for e in c.entities])
    elif t == "exactly_n":
        return Sum([If(vars[f"group_{e}"] == c.group, 1, 0) for e in c.entities]) == c.n
    elif t == "is_in":                                                                  # NEW
        return vars[f"group_{c.entity}"] == c.group

    # Logical wrappers
    elif t == "if_then": return Implies(encode(c.antecedent, vars), encode(c.consequent, vars))
    elif t == "not":     return Not(encode(c.claim, vars))
    elif t == "and":     return And(*[encode(cl, vars) for cl in c.claims])
    elif t == "or":      return Or(*[encode(cl, vars) for cl in c.claims])

    else:
        raise ValueError(f"Unknown constraint type: {t}")

def _collect_types(c) -> set[str]:
    t = c.type
    types = {t}
    if   t == "if_then":     types |= _collect_types(c.antecedent) | _collect_types(c.consequent)
    elif t == "not":         types |= _collect_types(c.claim)
    elif t in ("and", "or"):
        for cl in c.claims:  types |= _collect_types(cl)
    return types

def _used_types(extracted) -> set[str]:
    types = set()
    for c in extracted.constraints:
        types |= _collect_types(c)
    for q in extracted.questions:
        for c in q.question_constraints:
            types |= _collect_types(c)
        for ch in q.answer_choices:
            for c in ch.constraints:
                types |= _collect_types(c)
    return types

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 24.9 MB/s eta 0:00:00


In [ ]:
import torch
import json
from transformers import TrainerCallback
import gc


Z3_SAMPLE_SIZE = 50

def _get_z3_answer(json_str, active_domains):
    try:
        parsed     = json.loads(json_str)
        schema     = build_hybrid_schema(active_domains)
        logic_prob = schema(**parsed)
        result     = z3_solve(logic_prob)
        if result["status"] != "sat":
            return None
        return result["question_results"]
    except Exception:
        return None

class Z3AccuracyCallback(TrainerCallback):
    def __init__(self, val_raw_sample, tokenizer, model):
        self.sample = val_raw_sample
        self.tok    = tokenizer
        self.model  = model
        self.n      = len(val_raw_sample)

    def on_evaluate(self, args, state, control, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()

        FastLanguageModel.for_inference(self.model)
        parseable = z3_correct = z3_attempted = 0
        failures = []   # ADDED: ACCUMULATOR FOR PER-ROW FAILURES TO BE LOGGED
        try:
            for i, row in enumerate(self.sample):   # CHANGED: enumerate() SO ROW INDEX 'i' CAN BE RECORDED IN THE LOG
                active_domains = json.loads(row["active_domains"])
                user_content = (
                    f"Active domains: {', '.join(active_domains)}\n\n"
                    f"Extract this logic puzzle:\n\n{row['problem_text']}"
                )
                prompt = self.tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM},
                     {"role": "user",   "content": user_content}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=False,
                )
                inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)
                with torch.no_grad():
                    out = self.model.generate(
                        **inputs, max_new_tokens=2048, do_sample=False,
                        use_cache=True, pad_token_id=self.tok.eos_token_id,
                    )
                pred_text = self.tok.decode(
                    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
                ).strip()
                del inputs, out

                try:
                    json.loads(pred_text)
                    parseable += 1
                except json.JSONDecodeError:
                    # ADDED: RECORD PARSE FAILURE (MODEL OUTPUT WAS NOT VALID JSON)
                    failures.append({
                        "idx": i, "domains": active_domains, "failure": "parse",
                        "pred": pred_text, "expected": row["extracted_json"],
                        "problem_text": row["problem_text"],
                        "pred_ans": None, "true_ans": None,
                    })
                    continue

                truth_answer = _get_z3_answer(row["extracted_json"], active_domains)
                if truth_answer is None:
                    continue
                pred_answer = _get_z3_answer(pred_text, active_domains)
                z3_attempted += 1
                if pred_answer == truth_answer:
                    z3_correct += 1
                else:
                    # ADDED: RECORD SEMANTIC FAILURE (VALID JSON BUT Z3 ANSWER DISAGREES WITH TRUTH)
                    failures.append({
                        "idx": i, "domains": active_domains, "failure": "z3_wrong",
                        "pred": pred_text, "expected": row["extracted_json"],
                        "problem_text": row["problem_text"],
                        "pred_ans": str(pred_answer), "true_ans": str(truth_answer),
                    })
        finally:
            FastLanguageModel.for_training(self.model)
            self.model.config.use_cache = False
            gc.collect()
            torch.cuda.empty_cache()

        print(
            f"\n[Z3 Callback] step={state.global_step} | "
            f"parse={parseable}/{self.n} | "
            f"z3_acc={z3_correct}/{z3_attempted} attempted "
            f"({z3_correct}/{self.n} overall)\n"
        )

        # ADDED: DUMP ALL COLLECTED FAILURES TO A PER-STEP FILE ON DRIVE FOR INSPECTION
        if failures:
            failure_log = f"/content/drive/MyDrive/ns_proj/z3_new_failures_step{state.global_step}.txt"
            with open(failure_log, "w") as flog:
                for fail in failures:
                    flog.write(
                        f"\n[idx={fail['idx']} | failure={fail['failure']}"
                        f" | domains={fail['domains']} | step={state.global_step}]\n"
                    )
                    flog.write(f"PROBLEM:\n{fail['problem_text']}\n\n")
                    flog.write(f"PRED:\n{fail['pred']}\n\nEXPECTED:\n{fail['expected']}\n\n")
                    if fail["pred_ans"]:
                        flog.write(f"PRED_ANS:  {fail['pred_ans']}\nTRUE_ANS:  {fail['true_ans']}\n")
                    flog.write("-" * 40 + "\n")
            print(f"[Z3 Callback] Failures written to {failure_log}")

val_raw_sample = [
    {
        "problem_text":   ex["problem_text"],
        "active_domains": ex["active_domains"],
        "extracted_json": ex["extracted_json"],
    }
    for ex in ds_eval.select(range(min(Z3_SAMPLE_SIZE, len(ds_eval))))
]

z3_callback = Z3AccuracyCallback(val_raw_sample, tokenizer, model)
print(len(val_raw_sample))
print("=========")
print(val_raw_sample[0].keys())

50
dict_keys(['problem_text', 'active_domains', 'extracted_json'])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from transformers import trainer_utils


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_train,
    eval_dataset  = ds_eval,
    #callbacks     = [z3_callback],  # commented so training goes faster
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        per_device_eval_batch_size = 1,
        eval_accumulation_steps = 1,
        warmup_ratio = 0.03,
        lr_scheduler_type = "cosine",
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 1,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        report_to = "none",
        output_dir = "/content/drive/MyDrive/ns_proj/checkpoints", # in case script crashes it can resume from checkpoint
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

checkpoint = trainer_utils.get_last_checkpoint("/content/drive/MyDrive/ns_proj/checkpoints")
trainer.train(resume_from_checkpoint=checkpoint) # in case script crashes it can resume from checkpoint

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/777 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/138 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/777 [00:00<?, ? examples/s]

...

Map (num_proc=6):   0%|          | 0/138 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/138 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 777 | Num Epochs = 3 | Total steps = 147
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Step,Training Loss,Validation Loss


TrainOutput(global_step=147, training_loss=0.0, metrics={'train_runtime': 0.0108, 'train_samples_per_second': 215094.879, 'train_steps_per_second': 13564.542, 'total_flos': 7905168015360000.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

# 1. Clean up any leftover callbacks from previous crashes
while any(isinstance(cb, Z3AccuracyCallback) for cb in trainer.callback_handler.callbacks):
    trainer.remove_callback(Z3AccuracyCallback)

# 2. Prevent the NotebookProgressCallback from crashing the standalone evaluation
try:
    trainer.remove_callback(NotebookProgressCallback)
except ValueError:
    pass

# 3. Instantiate and add exactly ONE Z3 callback
z3_callback = Z3AccuracyCallback(val_raw_sample, tokenizer, model)
trainer.add_callback(z3_callback)

# 4. Create your small eval dataset from the Trainer's tokenized dataset
small_eval_dataset = trainer.eval_dataset.select(range(50))

# 5. Run the evaluation
trainer.evaluate(eval_dataset=small_eval_dataset)

# 6. Remove the callback afterwards so it doesn't linger
trainer.remove_callback(z3_callback)

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


[Z3 Callback] step=279 | parse=50/50 | z3_acc=16/16 attempted (16/50 overall)



In [ ]:
ex = trainer.train_dataset[0]
labels    = ex["labels"]
input_ids = ex["input_ids"]

masked = sum(1 for l in labels if l == -100)
print(f"{masked}/{len(labels)} tokens masked (should be most of them)\n")

# decode ONLY the tokens that contribute to loss:
kept = [i for i, l in zip(input_ids, labels) if l != -100]
print("LOSS IS COMPUTED ON:\n", tokenizer.decode(kept))

227/791 tokens masked (should be most of them)

LOSS IS COMPUTED ON:
 <think>

</think>

{"entities": ["Charles", "Ralph", "Angela"], "constraints": [{"type": "is_deceiver", "entity": "Charles", "evidence_text": "Charles is a deceiver (knave)."}, {"type": "is_truth_teller", "entity": "Ralph", "evidence_text": "Ralph is a truth-teller (knight)."}, {"type": "is_truth_teller", "entity": "Angela", "evidence_text": "Angela is a truth-teller (knight)."}, {"type": "if_then", "antecedent": {"type": "is_truth_teller", "entity": "Charles"}, "consequent": {"type": "is_deceiver", "entity": "Ralph"}, "evidence_text": "If Charles is a truth-teller (knight), then Ralph is a deceiver (knave)."}, {"type": "if_then", "antecedent": {"type": "is_deceiver", "entity": "Charles"}, "consequent": {"type": "not", "claim": {"type": "is_deceiver", "entity": "Ralph"}}, "evidence_text": "If Charles is a deceiver (knave), then it is not the case that: Ralph is a deceiver (knave)."}, {"type": "if_then", "antecedent":

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)   # 2x faster generate; switches model to inference mode

row = raw[0]
domains = json.loads(row["active_domains"])
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": f"Active domains: {', '.join(domains)}\n\n"
                                  f"Extract this logic puzzle:\n\n{row['problem_text']}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    enable_thinking=False, return_tensors="pt",
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False)  # greedy
gen = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("GENERATED:\n", gen)
print("\nEXPECTED:\n", row["extracted_json"])

json_str = gen[gen.index("{"):gen.rindex("}") + 1]   # strip the empty think block
print("\nEXACT MATCH:", json.loads(json_str) == json.loads(row["extracted_json"]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

GENERATED:
 {"entities": ["Helena", "Luther", "Thomas", "Quinn"], "constraints": [{"type": "before", "left": "Quinn", "right": "Helena", "evidence_text": "Quinn is before Helena."}, {"type": "immediately_before", "left": "Helena", "right": "Luther", "evidence_text": "Helena is immediately before Luther."}, {"type": "adjacent", "left": "Helena", "right": "Luther", "evidence_text": "Helena and Luther are adjacent."}, {"type": "adjacent", "left": "Helena", "right": "Quinn", "evidence_text": "Helena and Quinn are adjacent."}, {"type": "adjacent", "left": "Thomas", "right": "Quinn", "evidence_text": "Thomas and Quinn are adjacent."}, {"type": "if_then", "antecedent": {"type": "adjacent", "left": "Thomas", "right": "Quinn"}, "consequent": {"type": "before", "left": "Quinn", "right": "Luther"}, "evidence_text": "If Thomas and Quinn are adjacent, then Quinn is before Luther."}], "questions": [{"question_constraints": [], "answer_choices": [{"label": "A", "type": "could_be_true", "constraints":

 6 immediate cells below this are to export

In [ ]:
!pip install -q "torchao>=0.16.0"

In [ ]:
model.save_pretrained(MODEL_OUT_PATH)        # writes the GOOD adapter into qwen3_lora
tokenizer.save_pretrained(MODEL_OUT_PATH)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/ns_proj/qwen3_lora/tokenizer_config.json.


('/content/drive/MyDrive/ns_proj/qwen3_lora/tokenizer_config.json',
 '/content/drive/MyDrive/ns_proj/qwen3_lora/chat_template.jinja',
 '/content/drive/MyDrive/ns_proj/qwen3_lora/tokenizer.json')

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "unsloth/Qwen3-0.6B"
ADAPTER = "/content/drive/MyDrive/ns_proj/qwen3_lora"

# 1. Load base in FULL precision — no 4-bit
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cuda")
tok = AutoTokenizer.from_pretrained(BASE)

# 2. Load your LoRA adapter on top
model_merged = PeftModel.from_pretrained(base, ADAPTER)

# 3. Merge and unload — clean 16-bit merge, no rounding corruption
model_merged = model_merged.merge_and_unload()

# 4. Quick sanity check before saving
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to("cuda")
out = model_merged.generate(**inputs, max_new_tokens=2048, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AttributeError: 'Qwen3Attention' object has no attribute 'apply_qkv'

In [ ]:
model_merged.save_pretrained("/content/drive/MyDrive/ns_proj/gguf_clean")
tok.save_pretrained("/content/drive/MyDrive/ns_proj/gguf_clean")
print("saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved


In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements.txt

Cloning into '/tmp/llama.cpp'...
remote: Enumerating objects: 3334, done.
remote: Counting objects: 100% (3334/3334), done.
remote: Compressing objects: 100% (2687/2687), done.
remote: Total 3334 (delta 614), reused 2744 (delta 565), pack-reused 0 (from 0)
Receiving objects: 100% (3334/3334), 33.49 MiB | 19.56 MiB/s, done.
Resolving deltas: 100% (614/614), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 509.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 53.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!python /tmp/llama.cpp/convert_hf_to_gguf.py \
    /content/drive/MyDrive/ns_proj/gguf_clean \
    --outfile /content/drive/MyDrive/ns_proj/qwen3-logic-v4.Q8_0.gguf \
    --outtype q8_0

INFO:hf-to-gguf:Loading model: gguf_clean
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {1024, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> Q8_0, shape = {1024, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float1

In [ ]:
import hashlib
h = hashlib.sha256()
with open("/content/drive/MyDrive/ns_proj/qwen3-logic-v4.Q8_0.gguf", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print(h.hexdigest())

06a7c49f90b99dda739e2912e7414b288671acbeb5ec596ffc6771106dcf3978


Below is unnecessary just debugging

In [ ]:
# Ensure live model is in fast inference mode without re-allocating
FastLanguageModel.for_inference(model)

SYSTEM = "Extract logic puzzles into JSON. Return ONLY a JSON object, no explanation."

# Problem 5 from your benchmark logs (unseen by the previous live test cell)
prob5_text = """There are 5 positions numbered 1 through 5. Every person is either a knight who always tells the truth or a knave who always lies, and there are 3 distinct groups. Mona, Gwen, Quinn, Iris, and Sam occupy these positions.

Iris and Sam are not adjacent. Exactly 1 person from the set {Mona, Gwen, Quinn} is in group 3. Exactly 2 people from the set {Mona, Gwen, Sam} are in group 1.

If Mona is a knight, then Iris and Sam are not adjacent. If exactly 1 person from the set {Gwen, Quinn, Sam} is in group 3, then Gwen is immediately before Mona. If Iris is a knight, then exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Iris is a knave, then it is not the case that exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Gwen is a knight, then Sam is before Mona. If Gwen is a knave, then it is not the case that Sam is before Mona.

Which of the following is correct?
A) [must be true] Iris is in slot 1
B) [must be true] Iris is in slot 5
C) [could be true] Iris is in slot 4
D) [must be true] Iris is in slot 2"""

prob5_domains = ["grouping", "knights_and_knaves", "ordering"]

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"Active domains: {', '.join(prob5_domains)}\n\nExtract this logic puzzle:\n\n{prob5_text}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False)
gen = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("="*60)
print("LIVE MODEL GENERALIZATION TEST (PROBLEM 5):")
print("="*60)
print(gen)

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


LIVE MODEL GENERALIZATION TEST (PROBLEM 5):
<think>

</think>

{"entities": ["Mona", "Gwen", "Quinn", "Iris", "Sam"], "constraints": [{"type": "not", "claim": {"type": "adjacent", "left": "Iris", "right": "Sam"}, "evidence_text": "Iris and Sam are not adjacent."}, {"type": "exactly_n", "entities": ["Mona", "Gwen", "Quinn"], "n": 1, "group": 3, "evidence_text": "Exactly 1 person from the set {Mona, Gwen, Quinn} is in group 3."}, {"type": "exactly_n", "entities": ["Mona", "Gwen", "Sam"], "n": 2, "group": 1, "evidence_text": "Exactly 2 people from the set {Mona, Gwen, Sam} are in group 1."}], "questions": [{"question_constraints": [{"type": "if_then", "antecedent": {"type": "is_truth_teller", "entity": "Mona"}, "consequent": {"type": "not", "claim": {"type": "adjacent", "left": "Iris", "right": "Sam"}}, "evidence_text": "If Mona is a knight, then Iris and Sam are not adjacent."}, {"type": "if_then", "antecedent": {"type": "exactly_n", "entities": ["Gwen", "Quinn", "Sam"], "n": 1, "group":

In [ ]:
# (ideally restart runtime first so Unsloth's patches don't interfere; a 0.6B fp16 load is cheap)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


SYSTEM = "Extract logic puzzles into JSON. Return ONLY a JSON object, no explanation."

# Problem 5 from your benchmark logs (unseen by the previous live test cell)
prob5_text = """There are 5 positions numbered 1 through 5. Every person is either a knight who always tells the truth or a knave who always lies, and there are 3 distinct groups. Mona, Gwen, Quinn, Iris, and Sam occupy these positions.

Iris and Sam are not adjacent. Exactly 1 person from the set {Mona, Gwen, Quinn} is in group 3. Exactly 2 people from the set {Mona, Gwen, Sam} are in group 1.

If Mona is a knight, then Iris and Sam are not adjacent. If exactly 1 person from the set {Gwen, Quinn, Sam} is in group 3, then Gwen is immediately before Mona. If Iris is a knight, then exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Iris is a knave, then it is not the case that exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Gwen is a knight, then Sam is before Mona. If Gwen is a knave, then it is not the case that Sam is before Mona.

Which of the following is correct?
A) [must be true] Iris is in slot 1
B) [must be true] Iris is in slot 5
C) [could be true] Iris is in slot 4
D) [must be true] Iris is in slot 2"""

prob5_domains = ["grouping", "knights_and_knaves", "ordering"]

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"Active domains: {', '.join(prob5_domains)}\n\nExtract this logic puzzle:\n\n{prob5_text}"},
]

MERGED_16BIT = "/content/drive/MyDrive/ns_proj/gguf"   # Unsloth's merged-fp16 HF dir, pre-GGUF
m16 = AutoModelForCausalLM.from_pretrained(MERGED_16BIT, torch_dtype=torch.float16, device_map="cuda")
t16 = AutoTokenizer.from_pretrained(MERGED_16BIT)

prompt = t16.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = t16(prompt, return_tensors="pt").to("cuda")
out = m16.generate(**inputs, max_new_tokens=2048, do_sample=False)
print(t16.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] The tokenizer you are loading from '/content/drive/MyDrive/ns_proj/gguf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>

</think>

_constraints=[
    type_fixed_constraints(slot_fixed_constraints([slot_fixed_constraints(1, type_fixed_slot_fixed_slot_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixed_fixe

In [ ]:
#USED THIS FOR DEBUGGING - DONT NEED ANYMORE

import gc
import torch
from unsloth import FastLanguageModel

# 1. Obliterate the live model from VRAM so we are forced to read from disk
try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# 2. Reload strictly from the saved checkpoint folder
print(f"Reloading saved checkpoint from: {MODEL_OUT_PATH} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_OUT_PATH,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = False,
)
FastLanguageModel.for_inference(model)


# Problem 5 from your benchmark logs (unseen by the previous live test cell)
prob5_text = """There are 5 positions numbered 1 through 5. Every person is either a knight who always tells the truth or a knave who always lies, and there are 3 distinct groups. Mona, Gwen, Quinn, Iris, and Sam occupy these positions.

Iris and Sam are not adjacent. Exactly 1 person from the set {Mona, Gwen, Quinn} is in group 3. Exactly 2 people from the set {Mona, Gwen, Sam} are in group 1.

If Mona is a knight, then Iris and Sam are not adjacent. If exactly 1 person from the set {Gwen, Quinn, Sam} is in group 3, then Gwen is immediately before Mona. If Iris is a knight, then exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Iris is a knave, then it is not the case that exactly 2 people from the set {Mona, Gwen, Quinn, Iris} are in group 1. If Gwen is a knight, then Sam is before Mona. If Gwen is a knave, then it is not the case that Sam is before Mona.

Which of the following is correct?
A) [must be true] Iris is in slot 1
B) [must be true] Iris is in slot 5
C) [could be true] Iris is in slot 4
D) [must be true] Iris is in slot 2"""

prob5_domains = ["grouping", "knights_and_knaves", "ordering"]

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"Active domains: {', '.join(prob5_domains)}\n\nExtract this logic puzzle:\n\n{prob5_text}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False)
gen = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n" + "="*50)
print("SAVED DISK CHECKPOINT OUTPUT (PROBLEM 5):")
print("="*50)
print(gen)

Reloading saved checkpoint from: /content/drive/MyDrive/ns_proj/qwen3_lora ...
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

Unsloth: pad_token was a vision token (<|vision_pad|>) on a text-only model. Replaced with <|endoftext|> to avoid NaN losses.
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SAVED DISK CHECKPOINT OUTPUT (PROBLEM 4):
<think>

</think>

{"entities": ["Mona", "Gwen", "Quinn", "Iris", "Sam"], "constraints": [{"type": "not_adjacent", "left": "Iris", "right": "Sam"}, {"type": "exactly_n", "entities": ["Mona", "Gwen", "Quinn"], "n": 1, "group": 3}, {"type": "exactly_n", "entities": ["Mona", "Gwen", "Sam"], "n": 2, "group": 1}, {"type": "if_then", "antecedent": {"type": "is_truth_teller", "entity": "Mona"}, "consequent": {"type": "not_adjacent", "left": "Iris", "right": "Sam"}}, {"type": "if_then", "antecedent": {"type": "exactly_n", "entities": ["Gwen", "Quinn", "Sam"], "n": 1, "group": 3}, "consequent": {"type": "immediately_before", "left": "Gwen", "right": "Mona"}}, {"type": "if_then", "antecedent": {"type": "is_truth_teller", "entity": "Iris"}, "consequent": {"type": "exactly_n", "entities": ["Mona", "Gwen", "Quinn", "Iris"], "n": 2, "group": 1}}, {"type": "if_then", "antecedent": {"type": "is_deceiver", "entity": "Iris"}, "consequent": {"type": "not", "clai